In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import numpy as np
import pandas as pd


PROJECT   = Path("..")
DATA      = PROJECT / "data"
PROCESSED = DATA / "processed"
MP_DIR    = PROCESSED / "mediapipe"
OS_DIR    = PROCESSED / "opensmile"
WH_DIR    = PROCESSED / "whisper"

OUT_DIR    = DATA / "out" / "dataset"
REPORT_DIR = DATA / "out" / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SEGMENTS_WITH_SPLIT = OUT_DIR / "segments_with_split.csv"
TARGETS_MULTI       = OUT_DIR / "targets_multi.csv"


MIN_ROWS_FOR_STATS = 1
FLOAT32_OUT = True


def iqr(a: np.ndarray) -> float:
    a = np.asarray(a, float)
    return np.nanpercentile(a, 75) - np.nanpercentile(a, 25)

def ensure_float32(df: pd.DataFrame) -> pd.DataFrame:
    if not FLOAT32_OUT:
        return df
    for c in df.select_dtypes(include=[np.number]).columns:
        df[c] = df[c].astype(np.float32)
    return df

def summarize_vector(x: np.ndarray, prefix: str) -> dict:
    x = np.asarray(x, float)
    if x.size < MIN_ROWS_FOR_STATS or np.all(np.isnan(x)):
        return {
            f"{prefix}_mean": np.nan, f"{prefix}_median": np.nan, f"{prefix}_std": np.nan,
            f"{prefix}_iqr": np.nan,  f"{prefix}_min": np.nan,    f"{prefix}_max": np.nan,
            f"{prefix}_n": 0,         f"{prefix}_valid_pct": 0.0
        }
    return {
        f"{prefix}_mean":   float(np.nanmean(x)),
        f"{prefix}_median": float(np.nanmedian(x)),
        f"{prefix}_std":    float(np.nanstd(x, ddof=0)),
        f"{prefix}_iqr":    float(iqr(x)),
        f"{prefix}_min":    float(np.nanmin(x)),
        f"{prefix}_max":    float(np.nanmax(x)),
        f"{prefix}_n":      int(np.sum(~np.isnan(x))),
        f"{prefix}_valid_pct": float(np.mean(~np.isnan(x)))
    }


def interval_overlap(a0, a1, b0, b1) -> float:
    return max(0.0, min(a1, b1) - max(a0, b0))

def rowwise_overlap_fraction(rows_start, rows_end, seg_start, seg_end) -> np.ndarray:
    seg_len = max(1e-9, seg_end - seg_start)
    return np.maximum(0.0, np.minimum(rows_end, seg_end) - np.maximum(rows_start, seg_start)) / seg_len

def midpoint_in_segment(times: np.ndarray, seg_start: float, seg_end: float) -> np.ndarray:
    return (times >= seg_start) & (times <= seg_end)


def rising_edge_rate(x: np.ndarray, t: np.ndarray, thr: float) -> float:
    x = np.asarray(x, float).ravel()
    t = np.asarray(t, float).ravel()
    if x.size == 0 or t.size == 0: return np.nan
    n = min(len(x), len(t)); x, t = x[:n], t[:n]
    ok = np.isfinite(x) & np.isfinite(t)
    if ok.sum() < 2: return np.nan
    x, t = x[ok], t[ok]
    dur = float(max(1e-9, t[-1] - t[0]))
    b = (x > thr).astype(np.int8)
    edges = int(((b[1:] == 1) & (b[:-1] == 0)).sum())
    return edges / dur

def zero_cross_rate(x: np.ndarray, t: np.ndarray, amp_thr: float = 0.02) -> float:
    x = np.asarray(x, float).ravel()
    t = np.asarray(t, float).ravel()
    if x.size == 0 or t.size == 0: return np.nan
    n = min(len(x), len(t)); x, t = x[:n], t[:n]
    ok = np.isfinite(x) & np.isfinite(t)
    if ok.sum() < 2: return np.nan
    x, t = x[ok], t[ok]
    dur = float(max(1e-9, t[-1] - t[0]))
    xz = x.copy(); xz[np.abs(xz) < amp_thr] = 0.0
    s = np.sign(xz)
    zc = int(((s[1:] * s[:-1]) < 0).sum())
    return zc / dur


def has_data(*arrs) -> bool:
    for a in arrs:
        v = np.asarray(a, float)
        if np.isfinite(v).any():
            return True
    return False

def safe_mean_stack(arrs: list[np.ndarray]) -> np.ndarray:
    keep = []
    for a in arrs:
        v = np.asarray(a, float)
        if np.isfinite(v).any():
            keep.append(v)
    if not keep:
        return np.array([])
    return np.nanmean(np.vstack(keep), axis=0)


In [ ]:
segments = pd.read_csv(SEGMENTS_WITH_SPLIT)
print("Segments:", segments.shape)
segments.head(3)


In [ ]:
from glob import glob


BLENDSHAPES = [
    "eyeBlinkLeft","eyeBlinkRight","eyeSquintLeft","eyeSquintRight",
    "eyeLookUpLeft","eyeLookUpRight","eyeLookDownLeft","eyeLookDownRight",
    "eyeLookInLeft","eyeLookInRight","eyeLookOutLeft","eyeLookOutRight",
    "browInnerUp","browDownLeft","browDownRight","browOuterUpLeft","browOuterUpRight",
    "mouthOpen","jawOpen","cheekPuff","mouthSmileLeft","mouthSmileRight",
    "mouthFunnel","mouthPucker","mouthLeft","mouthRight",
    "mouthUpperUpLeft","mouthUpperUpRight","mouthLowerDownLeft","mouthLowerDownRight",
]

LANDMARKS = {
    "nose": ["nose_x","nose_y"],
    "leftEye": ["leftEye_x","leftEye_y"],
    "rightEye": ["rightEye_x","rightEye_y"],
    "leftShoulder": ["leftShoulder_x","leftShoulder_y"],
    "rightShoulder": ["rightShoulder_x","rightShoulder_y"],
    "leftWrist": ["leftWrist_x","leftWrist_y","leftWrist_visibility"],
    "rightWrist": ["rightWrist_x","rightWrist_y","rightWrist_visibility"],
}

def find_time_col(df: pd.DataFrame) -> str:
    preferred = ["timestamp_sec","t","time","timestamp","sec","seconds","time_s","time_sec","frameTime"]
    for c in preferred:
        if c in df.columns:
            return c
    for c in df.columns:
        lc = c.lower()
        if lc.endswith("_ms") or "millis" in lc:
            df[c] = pd.to_numeric(df[c], errors="coerce") / 1000.0
            return c
        if lc.endswith("_us") or "micro" in lc:
            df[c] = pd.to_numeric(df[c], errors="coerce") / 1_000_000.0
            return c
    raise ValueError("MediaPipe: time column not found (expected 'timestamp_sec' or similar).")

def available_cols(df: pd.DataFrame, names: list[str]) -> list[str]:
    return [c for c in names if c in df.columns]

def col_or_nan(df: pd.DataFrame, name: str) -> np.ndarray:
    return pd.to_numeric(df[name], errors="coerce").to_numpy(dtype=float) if name in df.columns else np.full(len(df), np.nan)

def norm_scale_per_frame(df: pd.DataFrame) -> np.ndarray:
    ls_ok = all(c in df.columns for c in ["leftShoulder_x","leftShoulder_y"])
    rs_ok = all(c in df.columns for c in ["rightShoulder_x","rightShoulder_y"])
    le_ok = all(c in df.columns for c in ["leftEye_x","leftEye_y"])
    re_ok = all(c in df.columns for c in ["rightEye_x","rightEye_y"])
    scale = np.full(len(df), np.nan, dtype=float)
    if ls_ok and rs_ok:
        dx = pd.to_numeric(df["leftShoulder_x"], errors="coerce").to_numpy(float) - pd.to_numeric(df["rightShoulder_x"], errors="coerce").to_numpy(float)
        dy = pd.to_numeric(df["leftShoulder_y"], errors="coerce").to_numpy(float) - pd.to_numeric(df["rightShoulder_y"], errors="coerce").to_numpy(float)
        scale = np.sqrt(dx*dx + dy*dy)
    if np.isnan(scale).all() and le_ok and re_ok:
        dx = pd.to_numeric(df["leftEye_x"], errors="coerce").to_numpy(float) - pd.to_numeric(df["rightEye_x"], errors="coerce").to_numpy(float)
        dy = pd.to_numeric(df["leftEye_y"], errors="coerce").to_numpy(float) - pd.to_numeric(df["rightEye_y"], errors="coerce").to_numpy(float)
        scale = np.sqrt(dx*dx + dy*dy)
    scale[scale <= 1e-3] = np.nan
    return scale

def head_track_xy(df: pd.DataFrame) -> tuple[np.ndarray,np.ndarray]:
    xs, ys = [], []
    for name in ["nose","leftEye","rightEye"]:
        pair = LANDMARKS.get(name, [])[:2]
        if len(pair)==2 and all(c in df.columns for c in pair):
            xs.append(pd.to_numeric(df[pair[0]], errors="coerce").to_numpy(float))
            ys.append(pd.to_numeric(df[pair[1]], errors="coerce").to_numpy(float))
    if not xs:
        n = len(df)
        return np.full(n, np.nan), np.full(n, np.nan)
    return np.nanmean(np.vstack(xs), axis=0), np.nanmean(np.vstack(ys), axis=0)

def dist(ax, ay, bx, by) -> np.ndarray:
    return np.sqrt((ax-bx)**2 + (ay-by)**2)

def load_mediapipe_tables(mp_dir: Path) -> dict[str, pd.DataFrame]:
    out = {}
    for p in sorted(mp_dir.glob("*.csv")):
        vid = p.stem.split("_")[0]
        df = pd.read_csv(p)
        if df.empty: 
            continue
        df["__video_id"] = vid
        out[vid] = df
    return out

def aggregate_mediapipe_for_video(vid: str, df: pd.DataFrame, segs_v: pd.DataFrame) -> pd.DataFrame:
    t_col = find_time_col(df)
    t = pd.to_numeric(df[t_col], errors="coerce").to_numpy(dtype=float)

    bs_cols = available_cols(df, BLENDSHAPES)
    bs = df[bs_cols] if bs_cols else pd.DataFrame(index=df.index)

    scale = norm_scale_per_frame(df)
    hx, hy = head_track_xy(df)

    
    cols = {c: col_or_nan(df, c) for c in set(sum(LANDMARKS.values(), []))}
    cols.setdefault("leftWrist_visibility",  np.full(len(df), np.nan))
    cols.setdefault("rightWrist_visibility", np.full(len(df), np.nan))

    rows = []
    for _, s in segs_v.iterrows():
        t0, t1 = float(s.t_start), float(s.t_end)
        seg_mask = midpoint_in_segment(t, t0, t1)
        seg_idx = np.where(seg_mask)[0]
        dur = max(1e-9, t1 - t0)
        row = {"video_id": vid, "seg_id": s.seg_id}

        
        if bs_cols:
            sub = bs.loc[seg_mask]

            
            b_left  = pd.to_numeric(sub.get("eyeBlinkLeft",  np.nan), errors="coerce").to_numpy(float) if "eyeBlinkLeft"  in sub else np.array([])
            b_right = pd.to_numeric(sub.get("eyeBlinkRight", np.nan), errors="coerce").to_numpy(float) if "eyeBlinkRight" in sub else np.array([])
            b_mean  = safe_mean_stack([b_left, b_right])
            if b_mean.size:
                blink_duty = float(np.mean(b_mean > 0.5))
                row["face__blink_duty"] = blink_duty
                row["face__blink_rate_hz"] = rising_edge_rate(b_mean, t[seg_mask], thr=0.5)
            else:
                row["face__blink_duty"] = np.nan
                row["face__blink_rate_hz"] = np.nan

            
            if "mouthOpen" in sub:
                m = pd.to_numeric(sub["mouthOpen"], errors="coerce").to_numpy(float)
                row.update(summarize_vector(m, "face__mouth_open"))
                row["face__mouth_open_duty"] = float((m > 0.3).mean()) if np.isfinite(m).any() else np.nan
            elif "jawOpen" in sub:
                m = 0.8 * pd.to_numeric(sub["jawOpen"], errors="coerce").to_numpy(float)
                row.update(summarize_vector(m, "face__mouth_open"))
                row["face__mouth_open_duty"] = float((m > 0.24).mean()) if np.isfinite(m).any() else np.nan
            else:
                row.update(summarize_vector(np.array([]), "face__mouth_open"))
                row["face__mouth_open_duty"] = np.nan

            
            if "jawOpen" in sub:
                j = pd.to_numeric(sub["jawOpen"], errors="coerce").to_numpy(float)
                row.update(summarize_vector(j, "face__jaw_open"))
            else:
                row.update(summarize_vector(np.array([]), "face__jaw_open"))

            
            sl = pd.to_numeric(sub.get("mouthSmileLeft",  np.nan), errors="coerce").to_numpy(float) if "mouthSmileLeft"  in sub else np.array([])
            sr = pd.to_numeric(sub.get("mouthSmileRight", np.nan), errors="coerce").to_numpy(float) if "mouthSmileRight" in sub else np.array([])
            sm = safe_mean_stack([sl, sr])
            asym = np.abs(sl - sr) if (sl.size and sr.size) else np.array([])
            row.update(summarize_vector(sm,   "face__smile"))
            row.update(summarize_vector(asym, "face__smile_asym"))
            row["face__smile_duty"] = float((sm > 0.5).mean()) if sm.size else np.nan

            
            br = safe_mean_stack([pd.to_numeric(sub.get(n, np.nan), errors="coerce").to_numpy(float)
                                  for n in ["browInnerUp","browOuterUpLeft","browOuterUpRight"] if n in sub])
            bf = safe_mean_stack([pd.to_numeric(sub.get(n, np.nan), errors="coerce").to_numpy(float)
                                  for n in ["browDownLeft","browDownRight"] if n in sub])
            row.update(summarize_vector(br, "face__brow_raise"))
            row.update(summarize_vector(bf, "face__brow_furrow"))

            
            row["blendshape__valid_frac"] = float(sub.notna().mean().mean())
        else:
            for k in ["face__blink_duty","face__blink_rate_hz","face__mouth_open_duty"]:
                row[k] = np.nan
            for k in ["face__mouth_open","face__jaw_open","face__smile","face__smile_asym","face__brow_raise","face__brow_furrow"]:
                row.update(summarize_vector(np.array([]), k))
            row["blendshape__valid_frac"] = 0.0

        
        if seg_idx.size >= 2 and not np.isnan(hx[seg_idx]).all():
            sc_all = scale[seg_idx]
            sc_ok  = np.isfinite(sc_all) & (sc_all > 1e-3)
            x = hx[seg_idx].astype(float); y = hy[seg_idx].astype(float)
            x[~sc_ok] = np.nan; y[~sc_ok] = np.nan
            x = x / sc_all;    y = y / sc_all
            vx = np.diff(x);   vy = np.diff(y)
            me = np.sqrt(vx*vx + vy*vy)
            row.update(summarize_vector(me, "face__head_motion"))
            if me.size and np.isfinite(me).any():
                med = np.nanmedian(me); p90 = np.nanpercentile(me, 90)
                row["face__head_motion_burst"] = float(p90/med) if med > 0 else np.nan
            else:
                row["face__head_motion_burst"] = np.nan
            tt = (t[seg_idx][1:] + t[seg_idx][:-1]) / 2.0
            vx = vx - np.nanmean(vx); vy = vy - np.nanmean(vy)
            row["face__nod_rate_hz"]   = zero_cross_rate(vy, tt, amp_thr=0.02)
            row["face__shake_rate_hz"] = zero_cross_rate(vx, tt, amp_thr=0.02)
        else:
            row.update(summarize_vector(np.array([]), "face__head_motion"))
            row["face__head_motion_burst"] = np.nan
            row["face__nod_rate_hz"] = np.nan
            row["face__shake_rate_hz"] = np.nan

        
        lwx, lwy = cols.get("leftWrist_x"), cols.get("leftWrist_y")
        rwx, rwy = cols.get("rightWrist_x"), cols.get("rightWrist_y")
        lvis     = cols.get("leftWrist_visibility")
        rvis     = cols.get("rightWrist_visibility")
        lsx, lsy = cols.get("leftShoulder_x"), cols.get("leftShoulder_y")
        rsx, rsy = cols.get("rightShoulder_x"), cols.get("rightShoulder_y")
        nx, ny   = cols.get("nose_x"), cols.get("nose_y")

        lwx, lwy = lwx[seg_mask], lwy[seg_mask]
        rwx, rwy = rwx[seg_mask], rwy[seg_mask]
        lvis, rvis = lvis[seg_mask], rvis[seg_mask]
        lsx, lsy = lsx[seg_mask], lsy[seg_mask]
        rsx, rsy = rsx[seg_mask], rsy[seg_mask]
        nx, ny   = nx[seg_mask],  ny[seg_mask]
        sc = scale[seg_mask]; sc_ok = np.isfinite(sc) & (sc > 1e-3)

        def norm_pair(ax, ay):
            if ax is None or ay is None or ax.size==0: 
                return np.array([]), np.array([])
            v = ax.astype(float).copy(); w = ay.astype(float).copy()
            v[~sc_ok] = np.nan; w[~sc_ok] = np.nan
            return v / sc, w / sc

        lwx_n, lwy_n = norm_pair(lwx, lwy)
        rwx_n, rwy_n = norm_pair(rwx, rwy)
        nx_n,  ny_n  = norm_pair(nx,  ny)

        
        if lwx_n.size and rwx_n.size:
            ws = dist(lwx_n, lwy_n, rwx_n, rwy_n)
            row.update(summarize_vector(ws, "pose__wrist_spread"))
        else:
            row.update(summarize_vector(np.array([]), "pose__wrist_spread"))

        
        if lsy.size and rsy.size and (lwy_n.size or rwy_n.size):
            sh_y = (lsy + rsy) / 2.0
            wh_y = np.nanmean(np.vstack([lwy, rwy]), axis=0)  
            hh = (wh_y / sc) - (sh_y / sc)                    
            row["pose__hands_height_mean"] = float(np.nanmean(hh)) if np.isfinite(hh).any() else np.nan
        else:
            row["pose__hands_height_mean"] = np.nan

        
        if nx_n.size and (lwx_n.size or rwx_n.size):
            d_l = dist(lwx_n, lwy_n, nx_n, ny_n) if lwx_n.size else np.full_like(nx_n, np.nan)
            d_r = dist(rwx_n, rwy_n, nx_n, ny_n) if rwx_n.size else np.full_like(nx_n, np.nan)
            dmin = np.nanmin(np.vstack([d_l, d_r]), axis=0)
            row["pose__hand_face_min_mean"] = float(np.nanmean(dmin)) if np.isfinite(dmin).any() else np.nan
            row["pose__hand_face_touch_rate"] = float(np.mean(dmin < 0.15)) if dmin.size else np.nan
        else:
            row["pose__hand_face_min_mean"] = np.nan
            row["pose__hand_face_touch_rate"] = np.nan

        
        vis_flags = []
        if lvis.size: vis_flags.append((lvis > 0.5).astype(float))
        if rvis.size: vis_flags.append((rvis > 0.5).astype(float))
        row["pose__vis_hand_frac"] = float(np.nanmean(np.vstack(vis_flags))) if vis_flags else np.nan

        
        pose_valid = []
        for n in ["leftWrist_x","leftWrist_y","rightWrist_x","rightWrist_y","leftShoulder_x","rightShoulder_x"]:
            if n in cols:
                pose_valid.append(~np.isnan(cols[n][seg_mask]))
        row["pose__valid_frac"] = float(np.nanmean(np.vstack(pose_valid))) if pose_valid else 0.0

        rows.append(row)

    df_out = ensure_float32(pd.DataFrame(rows))
    return df_out

def build_face_features(segments: pd.DataFrame, mp_dir: Path) -> pd.DataFrame:
    mp_tables = load_mediapipe_tables(mp_dir)
    parts = []
    missing = []
    for vid, segs_v in segments.groupby("video_id"):
        df = mp_tables.get(vid)
        if df is None or df.empty:
            missing.append(vid)
            parts.append(pd.DataFrame({"video_id": segs_v["video_id"], "seg_id": segs_v["seg_id"]}))
            continue
        parts.append(aggregate_mediapipe_for_video(vid, df, segs_v))
    feats = pd.concat(parts, ignore_index=True)

    keep_exact = [
        "video_id","seg_id",
        "face__blink_duty","face__blink_rate_hz",
        "face__mouth_open_mean","face__mouth_open_std","face__mouth_open_iqr","face__mouth_open_duty",
        "face__jaw_open_mean","face__jaw_open_std","face__jaw_open_iqr",
        "face__smile_mean","face__smile_std","face__smile_asym_mean","face__smile_duty",
        "face__brow_raise_mean","face__brow_raise_std","face__brow_furrow_mean","face__brow_furrow_std",
        "face__head_motion_mean","face__head_motion_std","face__head_motion_iqr","face__head_motion_burst",
        "face__nod_rate_hz","face__shake_rate_hz",
        "pose__wrist_spread_mean","pose__wrist_spread_std","pose__wrist_spread_iqr",
        "pose__hands_height_mean","pose__hand_face_min_mean","pose__hand_face_touch_rate",
        "pose__vis_hand_frac","blendshape__valid_frac","pose__valid_frac",
    ]
    keep_cols = [c for c in keep_exact if c in feats.columns]
    feats = feats[keep_cols].copy()

    feats.to_parquet(OUT_DIR / "features_face.parquet", index=False)
    if missing:
        print(f"[MediaPipe missing for videos")
    print(f" features_face.parquet written")
    return feats

face_feats = build_face_features(segments, MP_DIR)
face_feats.head(3)


In [ ]:
def load_opensmile_tables(os_dir: Path) -> dict:
    out = {}
    for p in sorted(os_dir.glob("*.csv")):
        vid = p.stem.split("_")[0]
        df = pd.read_csv(p)
        if df.empty:
            continue
        df["__video_id"] = vid
        out[vid] = df
    return out

def find_interval_cols(df: pd.DataFrame) -> tuple:
    c_start = c_end = None
    for c in df.columns:
        lc = c.lower()
        if lc in ("t_start","start","begin","onset","ts","start_time","from"):
            c_start = c
        if lc in ("t_end","end","offset","te","end_time","to"):
            c_end = c
    if not c_start or not c_end:
        raise ValueError("openSMILE: need interval columns (start/end).")
    return c_start, c_end


PRIMARY_KEEP = [
    "F0semitoneFrom27.5Hz_sma3nz_amean",
    "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
    "F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope",
    "F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope",
    "loudness_sma3_amean",
    "loudness_sma3_stddevNorm",
    "loudness_sma3_meanRisingSlope",
    "loudness_sma3_meanFallingSlope",
    "spectralFlux_sma3_amean",
    "spectralFlux_sma3_stddevNorm",
    
    "jitterLocal_sma3nz_amean",
    "HNRdBACF_sma3nz_amean",
]
MFCC_VOICED = [
    "mfcc1V_sma3nz_amean","mfcc1V_sma3nz_stddevNorm",
    "mfcc2V_sma3nz_amean","mfcc2V_sma3nz_stddevNorm",
    "mfcc3V_sma3nz_amean","mfcc3V_sma3nz_stddevNorm",
    "mfcc4V_sma3nz_amean","mfcc4V_sma3nz_stddevNorm",
]
MFCC_PLAIN = [
    "mfcc1_sma3_amean","mfcc1_sma3_stddevNorm",
    "mfcc2_sma3_amean","mfcc2_sma3_stddevNorm",
    "mfcc3_sma3_amean","mfcc3_sma3_stddevNorm",
    "mfcc4_sma3_amean","mfcc4_sma3_stddevNorm",
]

def select_opensmile_features(df_cols: list) -> list:
    keep = [c for c in PRIMARY_KEEP if c in df_cols]
    if any(c in df_cols for c in MFCC_VOICED):
        keep += [c for c in MFCC_VOICED if c in df_cols]
    else:
        keep += [c for c in MFCC_PLAIN if c in df_cols]
    return keep

def prune_summary_columns(df: pd.DataFrame,
                          keep=("mean","std","iqr"),
                          extras=("audio__coverage",)) -> pd.DataFrame:
    ok = {"video_id","seg_id"}
    for c in df.columns:
        if c in ok: 
            continue
        if "__" in c and any(c.endswith(f"_{suf}") for suf in keep):
            ok.add(c)
    for c in extras:
        if c in df.columns: ok.add(c)
    return df[[c for c in df.columns if c in ok]].copy()


SILENCE_DEFAULTS = {
    "loudness_sma3_amean": -80.0,
    "loudness_sma3_stddevNorm": 0.0,
    "loudness_sma3_meanRisingSlope": 0.0,
    "loudness_sma3_meanFallingSlope": 0.0,
    "spectralFlux_sma3_amean": 0.0,
    "spectralFlux_sma3_stddevNorm": 0.0,
    
}

SLOPE_FEATURES = [
    "F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope",
    "F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope",
    "loudness_sma3_meanRisingSlope",
    "loudness_sma3_meanFallingSlope",
]

def build_audio_features(segments: pd.DataFrame, os_dir: Path) -> pd.DataFrame:
    os_tables = load_opensmile_tables(os_dir)
    parts = []

    for vid, segs_v in segments.groupby("video_id"):
        df = os_tables.get(vid)

        if df is None or df.empty:
            parts.append(pd.DataFrame({"video_id": segs_v["video_id"], "seg_id": segs_v["seg_id"],
                                       "audio__coverage": 0.0}))
            continue

        
        c_start, c_end = find_interval_cols(df)
        starts = pd.to_numeric(df[c_start], errors="coerce").to_numpy(float)
        ends   = pd.to_numeric(df[c_end],   errors="coerce").to_numpy(float)

        feat_cols = select_opensmile_features(list(df.columns))
        if not feat_cols:
            parts.append(pd.DataFrame({"video_id": segs_v["video_id"], "seg_id": segs_v["seg_id"],
                                       "audio__coverage": 0.0}))
            continue

        X = {c: pd.to_numeric(df[c], errors="coerce").to_numpy(float) for c in feat_cols}

        rows = []
        for _, s in segs_v.iterrows():
            t0, t1 = float(s.t_start), float(s.t_end)
            w = rowwise_overlap_fraction(starts, ends, t0, t1)  
            wmask = w > 0
            row = {"video_id": vid, "seg_id": s.seg_id}

            
            coverage = float(np.clip(w.sum(), 0.0, 1.0))
            row["audio__coverage"] = coverage

            if not wmask.any():
                
                for c in feat_cols:
                    if c in SILENCE_DEFAULTS:
                        v = SILENCE_DEFAULTS[c]
                        row[f"audio__{c}_mean"] = float(v)
                        row[f"audio__{c}_std"]  = 0.0
                        row[f"audio__{c}_iqr"]  = 0.0
                    else:
                        row[f"audio__{c}_mean"] = np.nan
                        row[f"audio__{c}_std"]  = np.nan
                        row[f"audio__{c}_iqr"]  = np.nan
                rows.append(row)
                continue

            ww = w[wmask]
            for c in feat_cols:
                x = X[c][wmask]
                
                if np.isfinite(ww).any() and ww.sum() > 0:
                    mu = float(np.average(x, weights=ww))
                else:
                    mu = float(np.nanmean(x))
                sd  = float(np.nanstd(x, ddof=0))
                q75 = np.nanpercentile(x, 75)
                q25 = np.nanpercentile(x, 25)
                iq  = float(q75 - q25)
                row[f"audio__{c}_mean"] = mu
                row[f"audio__{c}_std"]  = sd
                row[f"audio__{c}_iqr"]  = iq

            rows.append(row)

        feats_v = ensure_float32(pd.DataFrame(rows))
        parts.append(feats_v)

    feats = pd.concat(parts, ignore_index=True)

    
    for base in SLOPE_FEATURES:
        for suf in ("mean","std","iqr"):
            col = f"audio__{base}_{suf}"
            if col in feats.columns:
                p99 = feats[col].quantile(0.99)
                feats[col] = feats[col].clip(upper=p99)
    
    feats = prune_summary_columns(feats, keep=("mean","std","iqr"), extras=("audio__coverage",))

    out_pq = OUT_DIR / "features_audio.parquet"
    feats.to_parquet(out_pq, index=False)
    print(f"[OK] features_audio.parquet written: {len(feats)} rows, {feats.shape[1]} cols.")
    return feats

audio_feats = build_audio_features(segments, OS_DIR)
audio_feats.head(3)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

def load_whisper_tables(wh_dir: Path) -> dict:
    out = {}
    for p in sorted(wh_dir.glob("*.csv")):
        vid = p.stem.split("_")[0]
        df = pd.read_csv(p)
        if df.empty:
            continue
        df["__video_id"] = vid
        out[vid] = df
    return out

def find_interval_cols(df: pd.DataFrame) -> tuple:
    c_start = c_end = None
    for c in df.columns:
        lc = c.lower()
        if lc in ("t_start","start","begin","onset","ts","start_time","from"):
            c_start = c
        if lc in ("t_end","end","offset","te","end_time","to"):
            c_end = c
    if not c_start or not c_end:
        raise ValueError("Whisper: need interval columns (start/end).")
    return c_start, c_end

def infer_text_cols(df: pd.DataFrame):
    c_start, c_end = find_interval_cols(df)
    txt = conf = ntok = None
    for c in df.columns:
        lc = c.lower()
        if txt is None and lc in ("text","transcript","utterance"):
            txt = c
        if conf is None and ("conf" in lc or "confidence" in lc):
            conf = c
        if ntok is None and (lc in ("n_tokens","n_words") or "num_tokens" in lc or "tokens" in lc):
            ntok = c
    return c_start, c_end, txt, conf, ntok

def build_text_features(segments: pd.DataFrame, wh_dir: Path) -> pd.DataFrame:
    wh_tables = load_whisper_tables(wh_dir)
    parts = []

    for vid, segs_v in segments.groupby("video_id"):
        df = wh_tables.get(vid)
        if df is None or df.empty:
            parts.append(pd.DataFrame({
                "video_id": segs_v["video_id"],
                "seg_id": segs_v["seg_id"],
                "text__coverage": 0.0,
                "text__has_text": 0.0,
                "text__token_rate": 0.0,
                "text__char_rate": 0.0,
                "text__avg_conf": np.nan,
            }))
            continue

        c_start, c_end, c_text, c_conf, c_ntok = infer_text_cols(df)
        if c_text is None:
            c_text = "__text"
            df[c_text] = ""
        df[c_text] = df[c_text].fillna("").astype(str)

        if c_ntok is None:
            c_ntok = "__num_tokens"
            df[c_ntok] = df[c_text].apply(lambda s: len(s.split()))
        df[c_ntok] = pd.to_numeric(df[c_ntok], errors="coerce").fillna(0).astype(float)

        starts = pd.to_numeric(df[c_start], errors="coerce").to_numpy(float)
        ends   = pd.to_numeric(df[c_end],   errors="coerce").to_numpy(float)
        confv  = (pd.to_numeric(df[c_conf], errors="coerce").to_numpy(float)
                  if c_conf else np.full(len(df), np.nan))
        txtlen = df[c_text].str.len().to_numpy(dtype=float)

        rows = []
        for _, s in segs_v.iterrows():
            t0, t1 = float(s.t_start), float(s.t_end)
            seg_dur = max(1e-9, t1 - t0)

            w = rowwise_overlap_fraction(starts, ends, t0, t1)  
            wmask = w > 0
            ww = w[wmask]

            row = {"video_id": vid, "seg_id": s.seg_id}
            
            row["text__coverage"] = float(np.clip(w.sum(), 0.0, 1.0))

            if wmask.any():
                toks = float(np.sum(df.loc[wmask, c_ntok].to_numpy(float) * ww))
                chars = float(np.sum(txtlen[wmask] * ww))
                conf  = float(np.nanmean(confv[wmask])) if np.isfinite(confv[wmask]).any() else np.nan
            else:
                toks = 0.0
                chars = 0.0
                conf = np.nan

            has_text = float((toks > 0) or (chars > 0))
            row["text__has_text"]   = has_text
            row["text__token_rate"] = toks / seg_dur
            row["text__char_rate"]  = chars / seg_dur
            row["text__avg_conf"]   = conf

            rows.append(row)

        feats_v = ensure_float32(pd.DataFrame(rows))
        parts.append(feats_v)

    text_feats = pd.concat(parts, ignore_index=True)

    
    for col in ("text__token_rate","text__char_rate"):
        if col in text_feats.columns:
            p99 = float(text_feats[col].quantile(0.99))
            text_feats[col] = text_feats[col].clip(upper=p99)

    out_pq = OUT_DIR / "features_text.parquet"
    text_feats.to_parquet(out_pq, index=False)
    print(f"features_text.parquet written")
    return text_feats


text_feats = build_text_features(segments, WH_DIR)
text_feats.head(3)
